# Product Review Sentiment Analysis

A beginner-friendly Applied Deep Learning project for Central University of Haryana (CUH). This notebook uses real Amazon mobile-phone reviews and predicts Positive or Negative sentiment.

## 1. Import Libraries

Pandas loads and cleans the CSV. NumPy is useful for numerical work. Matplotlib and Seaborn will create charts in the next step.

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', 200)

## 2. Load Dataset

The public Amazon Unlocked Mobile Phones Reviews dataset has a `Reviews` column for customer text and a `Rating` column from 1 to 5.

In [ ]:
DATA_PATH = 'data/reviews.csv'

reviews_df = pd.read_csv(DATA_PATH)
print('Dataset shape:', reviews_df.shape)
print('Columns:', reviews_df.columns.tolist())
reviews_df.head()

## 3. Data Cleaning

We keep only the review text and rating because they are needed for sentiment analysis. We remove missing and duplicate reviews. Ratings 1–2 become Negative and ratings 4–5 become Positive. Rating 3 is excluded because it is neutral or mixed and this project uses binary classification.

In [ ]:
# Keep only the columns needed for this project and give them simple names.
reviews_df = reviews_df[['Reviews', 'Rating']].copy()
reviews_df.columns = ['review_text', 'rating']

print('Missing values before cleaning:')
print(reviews_df.isnull().sum())
print('Duplicate review rows before cleaning:', reviews_df.duplicated(subset='review_text').sum())
reviews_df.head()

In [ ]:
# Remove rows without review text or rating.
reviews_df = reviews_df.dropna(subset=['review_text', 'rating'])

# The same review should not train the model more than once.
reviews_df = reviews_df.drop_duplicates(subset='review_text')

# Keep only clear negative (1–2) and positive (4–5) ratings.
reviews_df = reviews_df[reviews_df['rating'].isin([1, 2, 4, 5])].copy()

# Convert rating to the binary target used by the models: 0 = Negative, 1 = Positive.
reviews_df['sentiment'] = (reviews_df['rating'] >= 4).astype(int)

print('Shape after cleaning:', reviews_df.shape)
print(reviews_df['sentiment'].value_counts().rename({0: 'Negative', 1: 'Positive'}))

### Simple text preprocessing

The function changes text to lowercase, removes links, punctuation, special characters, and extra spaces. It does **not** remove words such as `not`, `no`, or `never`, because they can change a review’s meaning.

In [ ]:
def clean_text(text):
    """Return a simple cleaned version of one review."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

reviews_df['clean_review'] = reviews_df['review_text'].apply(clean_text)
reviews_df = reviews_df[reviews_df['clean_review'].str.len() > 0].copy()
reviews_df[['review_text', 'clean_review', 'rating', 'sentiment']].head()

### Create a manageable working dataset

The original file has more than 400,000 reviews, which can be slow on a normal laptop. We use at most 50,000 cleaned reviews and sample the two sentiment classes equally. This keeps training practical while using genuine reviews. `random_state` makes the selected sample repeatable.

In [ ]:
MAX_REVIEWS = 50000
reviews_per_class = min(MAX_REVIEWS // 2, reviews_df['sentiment'].value_counts().min())

negative_reviews = reviews_df[reviews_df['sentiment'] == 0].sample(n=reviews_per_class, random_state=42)
positive_reviews = reviews_df[reviews_df['sentiment'] == 1].sample(n=reviews_per_class, random_state=42)

cleaned_df = pd.concat([negative_reviews, positive_reviews], ignore_index=True)
cleaned_df = cleaned_df.sample(frac=1, random_state=42).reset_index(drop=True)
cleaned_df['review_length'] = cleaned_df['clean_review'].str.split().str.len()

CLEANED_DATA_PATH = 'data/cleaned_reviews.csv'
cleaned_df.to_csv(CLEANED_DATA_PATH, index=False)

print('Final cleaned dataset shape:', cleaned_df.shape)
print(cleaned_df['sentiment'].value_counts().rename({0: 'Negative', 1: 'Positive'}))
print(f'Cleaned data saved to: {CLEANED_DATA_PATH}')
cleaned_df.head()

### What was done in Step 2?

We removed incomplete and duplicate reviews, excluded neutral ratings, created Positive/Negative labels, and cleaned text without removing important negation words. The result is a balanced, manageable dataset saved as `data/cleaned_reviews.csv`. Next, we will explore this cleaned dataset with five simple charts.